# 2. Train a TopK SAE from your saved activations

**Run this after notebook 1 has finished. No code changes are needed for your first run.**

1. Open in **Google Colab in your browser** and select **Runtime → Change runtime type → T4 GPU** (a larger GPU also works).
2. Choose **Runtime → Run all**, and connect the **same Google Drive account** as notebook 1.
3. The notebook finds your activations, trains the SAE, and saves the model and loss plot to Drive.

The authors' TopK SAE trains on **24 monthly snapshots** from 2017 and 2018, with **983,088 training vectors** in total. It checks reconstruction on **3 other dates** (122,886 validation vectors).

The defaults are 4,096 features, at most 32 active per input, and five passes through the training vectors. Step 5 runs the training loop.

## Step 1 — Connect Drive and load the SAE class

The folder matches notebook 1 automatically. Run both cells below.

In [ ]:
# @title Connect Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_FOLDER = Path('/content/drive/MyDrive/AMS691_GraphCast/monthly_2017_2018')
ACTIVATION_DIR = PROJECT_FOLDER / 'activations_layer8'
ACTIVATION_DIR.mkdir(parents=True, exist_ok=True)
print('Your project folder:', PROJECT_FOLDER)

In [ ]:
# @title Load the authors' SAE class
import importlib.util
import hashlib
import json
import os
import sys
import urllib.request

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Download only the unchanged SAE model file; training does not require JAX.
MODEL_URL = 'https://raw.githubusercontent.com/observerhan/graphcast-interpretability/470e1b9e1d1b432597191a2fa91b3f16c78c6a72/src/graphcast_interpretability/model.py'
MODEL_FILE = Path('/content/ams691_author_sae.py')
urllib.request.urlretrieve(MODEL_URL, MODEL_FILE)
if hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest() != '752ab5d7373a72ca8490ccc182ecdd717e533c8fc5676751fc8166caee503901':
    raise RuntimeError('Downloaded SAE source does not match the expected version.')
spec = importlib.util.spec_from_file_location('ams691_author_sae', MODEL_FILE)
author_sae = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = author_sae
spec.loader.exec_module(author_sae)
SAE = author_sae.SAE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('Choose Runtime > Change runtime type > T4 GPU, then run again.')
print('Training GPU:', torch.cuda.get_device_name())

## Step 2 — Training settings (you can keep all defaults)

An **epoch** is one pass through the training data. To continue a completed run, increase `EPOCHS` and run the notebook again.
To start a separate experiment, change `RUN_NAME`, for example to `topk_second_try`.

In [ ]:
# @title Training settings
EPOCHS = 5
RUN_NAME = 'topk_baseline'

N_FEATURES = 4096
K_ACTIVE = 32
BATCH_SIZE = 256
LEARNING_RATE = 3e-4
SEED = 0

if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {'.', '..'}:
    raise ValueError('RUN_NAME should be a simple folder name.')
if EPOCHS < 1 or not 1 <= K_ACTIVE <= N_FEATURES or BATCH_SIZE < 1 or LEARNING_RATE <= 0:
    raise ValueError('Use positive settings, with K_ACTIVE no larger than N_FEATURES.')
RUN_DIR = PROJECT_FOLDER / 'topk_runs' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Your results will be saved in:', RUN_DIR)

## Step 3 — Load the training and validation snapshots

This cell reads the dates that notebook 1 saved in `dataset.json`. It prints the training dates and the separate validation dates. By default, 24 dates train the model and 3 dates check it. It also copies those files to Colab's temporary storage for faster training.

In [ ]:
# @title Load the saved dataset
from datetime import datetime
import shutil

manifest_path = ACTIVATION_DIR / 'dataset.json'
if not manifest_path.exists():
    raise FileNotFoundError('Run notebook 1 first, using the same Google Drive account.')
dataset_manifest = json.loads(manifest_path.read_text())
expected_source = dataset_manifest['source']
if expected_source.get('layer') != 8 or expected_source.get('shape') != [40962, 512] or expected_source.get('dtype') != 'float32':
    raise ValueError('These files do not match the dataset produced by notebook 1.')

# Only the snapshots listed by notebook 1 belong to this experiment.
dates_by_file = {}
for center in sorted(dataset_manifest['requested_centers']):
    day = datetime.strptime(center, '%Y-%m-%dT%H').date().isoformat()
    path = ACTIVATION_DIR / f'layer0008_t{center}.npy'
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path.name}. Rerun notebook 1 to finish extraction.')
    dates_by_file[path] = day

days = sorted(set(dates_by_file.values()))
if len(days) < 2:
    raise ValueError('Collect at least two dates in notebook 1.')
# Older copies of notebook 1 used the last requested date for validation.
validation_days = sorted(set(dataset_manifest.get('validation_dates', [days[-1]])))
if not validation_days or not set(validation_days) < set(days):
    raise ValueError('The date list must include separate training and validation snapshots.')
training_days = [day for day in days if day not in validation_days]

cache = Path('/content/ams691_sae_data')
cache.mkdir(parents=True, exist_ok=True)
train_files, val_files, data_signature = [], [], []

for path, day in tqdm(dates_by_file.items(), desc='Reading activations from Drive'):
    local = cache / path.name
    temporary = local.with_suffix('.npy.tmp')
    shutil.copyfile(path, temporary)
    temporary.replace(local)
    array = np.load(local, mmap_mode='r', allow_pickle=False)
    if array.shape != (40962, 512) or array.dtype != np.float32 or not np.isfinite(array).all():
        raise ValueError(f'{path.name} is not a valid activation file from notebook 1.')
    with local.open('rb') as handle:
        digest = hashlib.file_digest(handle, 'sha256').hexdigest()
    data_signature.append({'file': path.name, 'sha256': digest})
    if day in validation_days:
        val_files.append(local)
    else:
        train_files.append(local)

print(f'Training snapshots ({len(train_files)}):', ', '.join(training_days))
print(f'Validation snapshots ({len(val_files)}):', ', '.join(validation_days))
print(f'Training vectors: {len(train_files) * 40962:,}; validation vectors: {len(val_files) * 40962:,}')

## Step 4 — Prepare the training calculation

Run this unchanged. The SAE tries to reconstruct each activation vector using a small number of features.
The reconstruction target uses the same centering and normalization as the authors' SAE.
An auxiliary loss helps features that have stopped activating; decoder vectors are kept at unit length.

The notebook also prepares automatic saving and continuation from the last completed epoch.

In [ ]:
# @title Prepare batches, loss, and saving (run unchanged)
import csv

def batches(paths, *, shuffle=False, epoch=0):
    rng = np.random.default_rng(SEED + epoch)
    order = list(paths)
    if shuffle:
        rng.shuffle(order)
    for path in order:
        array = np.load(path, mmap_mode='r', allow_pickle=False)
        indices = rng.permutation(len(array)) if shuffle else np.arange(len(array))
        for start in range(0, len(array), BATCH_SIZE):
            x = np.array(array[indices[start:start + BATCH_SIZE]], copy=True)
            yield torch.from_numpy(x).to(device)

def normalized_target(x):
    x = x - x.mean(dim=1, keepdim=True)
    return x / x.norm(dim=1, keepdim=True).clamp_min(1e-6)

@torch.no_grad()
def evaluate(model):
    model.eval()
    error = baseline_error = active = count = 0
    used = torch.zeros(N_FEATURES, dtype=torch.bool, device=device)
    for x in batches(val_files):
        target = normalized_target(x)
        reconstruction, code, _ = model(x)
        if not torch.isfinite(reconstruction).all():
            raise ValueError('Validation produced nonfinite values.')
        error += (target - reconstruction).square().sum().item()
        baseline_error += (target - train_mean).square().sum().item()
        active += (code > 0).sum().item()
        used |= (code > 0).any(dim=0)
        count += len(x)
    return {'validation_error': error / count,
            'mean_baseline_error': baseline_error / count,
            'fraction_variance_unexplained': error / baseline_error if baseline_error > 0 else None,
            'mean_active_features': active / count,
            'features_used_on_validation': int(used.sum().item())}

def save_json(path, value):
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    temporary.replace(path)

def save_checkpoint(name, epoch):
    temporary = RUN_DIR / (name + '.tmp')
    torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
                'config': model_config, 'data_signature': data_signature, 'validation_dates': validation_days,
                'epoch': epoch, 'best_error': best_error, 'best_epoch': best_epoch,
                'train_mean': train_mean.cpu(), 'history': history}, temporary)
    temporary.replace(RUN_DIR / name)

def save_history():
    temporary = RUN_DIR / 'history.csv.tmp'
    with temporary.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    temporary.replace(RUN_DIR / 'history.csv')

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
model_config = dict(d_in=512, latent=N_FEATURES, k=K_ACTIVE,
                    k_aux=min(512, N_FEATURES), dead_window=3_000_000,
                    aux_weight=1/32, batch_size=BATCH_SIZE, lr=LEARNING_RATE, seed=SEED,
                    normalization='per-row center, then L2 norm with epsilon 1e-6')
model = SAE(d_in=512, latent=N_FEATURES, k_active=K_ACTIVE,
            k_aux=model_config['k_aux'], dead_window=model_config['dead_window']).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

last_path = RUN_DIR / 'last.pt'
if last_path.exists():
    saved = torch.load(last_path, map_location=device, weights_only=True)
    if (saved['config'] != model_config or saved['data_signature'] != data_signature
            or saved.get('validation_dates', [days[-1]]) != validation_days):
        raise ValueError('Data, validation dates, or settings changed. Choose a new RUN_NAME in Step 2.')
    model.load_state_dict(saved['model_state'])
    optimizer.load_state_dict(saved['optimizer_state'])
    train_mean = saved['train_mean'].to(device)
    history = saved['history']
    best_error, best_epoch = saved['best_error'], saved['best_epoch']
    start_epoch = saved['epoch']
    print(f'Continuing after epoch {start_epoch}.')
else:
    if any(RUN_DIR.iterdir()):
        raise ValueError('This run folder contains files but no last checkpoint. Choose a new RUN_NAME.')
    # This baseline and the initial bias use TRAINING data only.
    total = torch.zeros(512, device=device, dtype=torch.float64)
    count = 0
    for x in batches(train_files):
        total += normalized_target(x).double().sum(dim=0)
        count += len(x)
    train_mean = (total / count).float()
    with torch.no_grad():
        model.b_pre.copy_(train_mean)
    initial = evaluate(model)
    history = [dict(epoch=0, training_error=None, auxiliary_error=None, **initial)]
    best_error, best_epoch, start_epoch = initial['validation_error'], 0, 0
    save_checkpoint('best.pt', 0)
    save_checkpoint('last.pt', 0)

save_json(RUN_DIR / 'config.json', {**model_config, 'requested_epochs': EPOCHS})
save_json(RUN_DIR / 'data_split.json', {
    'training_dates': training_days, 'validation_dates': validation_days,
    'training_files': [p.name for p in train_files],
    'validation_files': [p.name for p in val_files],
    'source': expected_source, 'data_signature': data_signature,
    'torch_version': str(torch.__version__), 'numpy_version': str(np.__version__),
})
print(f"Validation error before training: {history[0]['validation_error']:.6f}")

## Step 5 — Train the TopK SAE

**This is where training happens.** Each batch updates the encoder, decoder, and bias.
At the end of every epoch, the notebook checks validation error and saves:

- `best.pt`: the model with the lowest validation error.
- `last.pt`: the most recent model and optimizer, so training can continue.
- `history.csv`: the learning measurements.

If Colab disconnects, run this notebook again with the same settings. Training continues from the last completed epoch. An interrupted partial epoch is repeated.

In [ ]:
# @title Train and save to Google Drive
for epoch in range(start_epoch + 1, EPOCHS + 1):
    model.train()
    total_error = total_aux = count = 0
    n_batches = sum((len(np.load(path, mmap_mode='r')) + BATCH_SIZE - 1) // BATCH_SIZE for path in train_files)
    for x in tqdm(batches(train_files, shuffle=True, epoch=epoch), total=n_batches, desc=f'Epoch {epoch}/{EPOCHS}'):
        target = normalized_target(x)
        optimizer.zero_grad(set_to_none=True)
        reconstruction, code, auxiliary_reconstruction = model(x)
        reconstruction_loss = (target - reconstruction).square().sum(dim=1).mean()
        auxiliary_loss = reconstruction_loss.new_zeros(())
        if model.dead_mask.any():
            residual = (target - reconstruction).detach()
            auxiliary_loss = (residual - auxiliary_reconstruction).square().sum(dim=1).mean()
        loss = reconstruction_loss + model_config['aux_weight'] * auxiliary_loss
        if not torch.isfinite(loss):
            raise ValueError('Training loss is not finite. The last completed epoch remains saved.')
        loss.backward()
        author_sae._project_decoder_grads_orthogonal(model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'), error_if_nonfinite=True)
        optimizer.step()
        author_sae._renorm_decoder_columns_(model)

        # Same dead-feature tracking as the authors, keeping masks on the GPU.
        with torch.no_grad():
            active = (code > 0).any(dim=0)
            model.miss_counts.add_(len(x))
            model.miss_counts.masked_fill_(active, 0)
            model.dead_mask.copy_(model.miss_counts >= model.dead_window)
        total_error += reconstruction_loss.item() * len(x)
        total_aux += auxiliary_loss.item() * len(x)
        count += len(x)

    metrics = evaluate(model)
    history.append(dict(epoch=epoch, training_error=total_error/count, auxiliary_error=total_aux/count, **metrics))
    improved = metrics['validation_error'] < best_error
    if improved:
        best_error, best_epoch = metrics['validation_error'], epoch
        save_checkpoint('best.pt', epoch)
    save_checkpoint('last.pt', epoch)
    save_history()
    print(f"Epoch {epoch}: validation error = {metrics['validation_error']:.6f}; average active features = {metrics['mean_active_features']:.1f}")

save_history()
print(f'Finished. Best epoch: {best_epoch}. Results are saved in {RUN_DIR}')

## Step 6 — See the learning curve and load your best SAE

Lower reconstruction error is better. The dashed line reconstructs every validation vector with the **training mean**; a useful SAE should beat that simple baseline.
The active-feature count should be at most 32 with the default settings (it can be smaller because negative scores are set to zero).

Each point on the validation curve averages reconstruction error over all the held-out locations and dates. With the defaults, each of the three dates contributes equally. For the later cyclone F1 comparison, use separate storm dates from 2019–2021 with ClimateNetLarge labels.

In [ ]:
# @title Plot results and load the best model
best = torch.load(RUN_DIR / 'best.pt', map_location=device, weights_only=True)
model.load_state_dict(best['model_state'])
model.eval()
metrics = evaluate(model)

fig, ax = plt.subplots(figsize=(8, 4))
trained = [row for row in history if row['training_error'] is not None]
ax.plot([r['epoch'] for r in trained], [r['training_error'] for r in trained], 'o-', label='Training')
ax.plot([r['epoch'] for r in history], [r['validation_error'] for r in history], 'o-', label='Validation')
ax.axhline(metrics['mean_baseline_error'], color='gray', linestyle='--', label='Training-mean baseline')
ax.set(xlabel='Epoch (0 = before training)', ylabel='Squared reconstruction error per vector', title='TopK SAE learning curve')
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'training_curve.png', dpi=160)
plt.show()

summary = {'best_epoch': best['epoch'], 'validation_dates': validation_days, **metrics}
if len(validation_days) == 1:
    summary['validation_day'] = validation_days[0]  # Keep the original one-date summary format usable.
save_json(RUN_DIR / 'summary.json', summary)
print(f"Best model: epoch {best['epoch']}")
print(f"Validation error: {metrics['validation_error']:.6f}")
print(f"Training-mean baseline error: {metrics['mean_baseline_error']:.6f}")
print(f"Average active features: {metrics['mean_active_features']:.2f}")
if best['epoch'] == 0:
    print('Training has not improved on the initial model yet. Try more epochs before interpreting its features.')
print('\nSaved model:', RUN_DIR / 'best.pt')
print('Saved plot:', RUN_DIR / 'training_curve.png')
print('The variable model now contains your best SAE, ready for later feature analysis.')

You can close Colab now. The activations, model, and plot are saved in **My Drive → AMS691_GraphCast → monthly_2017_2018**.

This is your newly trained SAE. Its feature numbers do not automatically match the paper's pretrained SAE (for example, feature 3243 need not be the hurricane feature).